In [1]:
import math
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import sys
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp 
from numpy import linalg as LA
import import_ipynb
import helper_function
from helper_function import *

In [2]:
# For k = 0
def BurgersKEqualsZeroEqn(u, delta_x):
    ''' Calculate the equation for k = 0 in a system M

    Parameters
    ----------
    u: (M+1) x N matrix. 
        The matrix at a given time t. Here u[k,:] = u_k, for k in {0,...,M}
    delta_x: float
        Step size in the x-axis. Can also be understood as the spacing between each grid point

    Returns
    -------
    EqnKEqualsZero: array of length N
        One-dimensional array of length N. Representing u_0 
    '''
    num_vars = u.shape[0]
    dudx = getallxderiv(u, delta_x) # Get derivatives of all entries in matrix u using getallxderiv function
    # Compute equation
    if num_vars == 1:
        EqnKEqualsZero = -u[0,:] * dudx[0,:]
    # For M = 1 (number of variables is 2: u0 and u1)
    if num_vars == 2:
        EqnKEqualsZero = -u[0,:] * dudx[0,:] - u[1,:] * dudx[1,:]
    # For M = 2(u0, u1, u2)
    elif num_vars == 3:
        EqnKEqualsZero = -u[0,:] * dudx[0,:] - u[1,:] * dudx[1,:] - u[2,:] * dudx[2,:]

    return EqnKEqualsZero

In [3]:
# For k = 1
def BurgersKEqualsOneEqn(u, delta_x):
    ''' Calculate the equation for k = 0 in a M-system 
    
    Parameters
    ----------
    u: (M+1) x N matrix. 
        The matrix at a given time t. Here u[k,:] = u_k, for k in {0,...,M}
    delta_x: float
        Step size in the x-axis. Can also be understood as the spacing between each grid point

    Returns
    -------
    EqnKEqualsOne: array of length N
        One-dimensional array of length N. Representing u_1 
    '''
    # get the number of rows in the matrix
    num_vars = u.shape[0]
    dudx = getallxderiv(u, delta_x)
    breakpoint()
    # Compute equation
    # For M = 1
    if num_vars == 2:
        EqnKEqualsOne = -u[0,:] * dudx[1,:] - u[1,:] * dudx[0,:]
        
    # For M = 2
    elif num_vars == 3:
        EqnKEqualsOne = -u[1,:] * dudx[0,:] - (u[0,:] + 2*u[2,:]) * dudx[1,:] - 2*u[1,:] * dudx[2,:]
        
    return EqnKEqualsOne

In [4]:
# For k = 2
def BurgersKEqualsTwoEqn(u, delta_x):
    ''' Calculate the equation for k = 0 in a M-system 
    
    Parameters
    ----------
    u: (M+1) x N matrix. 
        The matrix at a given time t. Here u[k,:] = u_k, for k in {0,...,M}
    delta_x: float
        Step size in the x-axis. Can also be understood as the spacing between each grid point

    Returns
    -------
    EqnKEqualsTwo: array of length N
        One-dimensional array of length N. Representing u_2 
    '''
    num_vars = u.shape[0]
    dudx = getallxderiv(u, delta_x)
    
    # Compute equation for M=2, k=2
    EqnKEqualsTwo = -u[2,:] * dudx[0,:] - u[1,:] * dudx[1,:] - (u[0,:] + 4*u[2,:]) * dudx[2,:]
    return EqnKEqualsTwo

In [5]:
# General function
def GeneralBurgRHS(u, t, delta_x, M, N, ):
    ''' Calculate the solution matrix u in a M-system 
    
    Parameters
    ----------
    u: (M+1) x N matrix. 
        The solution matrix at a given time t. Here u[k,:] = u_k, for k in {0,...,M}
    t: vector
        Time vector
    delta_x: float
        Step size in the x-axis. Can also be understood as the spacing between each grid point
    M: non-negative integer
        Truncation limit. For now, the function is limited to M = 1 and M = 2
    N: non-negative integer
        Number of grid points (x-axis) in which we evaluate the solution
        
    Returns
    -------
    EqnM: array of length (M+1)*N
        One-dimensional (flattened) array of length (M+1)*N. We flatten it to put into ODE solver.
    '''
    # Reshape solution u to (M+1) x N matrix
    u = u.reshape(M+1, N)

    # For M = 0
    if M == 0:
        EqnKEqualsZero = BurgersKEqualsZeroEqn(u, delta_x)
        EqnM = EqnKEqualsZero.flatten()
        breakpoint()
        
    # For M = 1
    elif M == 1:
        EqnKEqualsZero = BurgersKEqualsZeroEqn(u, delta_x)
        EqnKEqualsOne = BurgersKEqualsOneEqn(u, delta_x)
        # Put into a (M+1) x N matrix
        EqnMEqualsOne = np.array([EqnKEqualsZero, EqnKEqualsOne])
        # Flatten for ODE solver
        EqnM = EqnMEqualsOne.flatten()

    # For M = 2
    elif M == 2:
        EqnKEqualsZero = BurgersKEqualsZeroEqn(u, delta_x)
        EqnKEqualsOne = BurgersKEqualsOneEqn(u, delta_x)
        EqnKEqualsTwo = BurgersKEqualsTwoEqn(u, delta_x)
        # Put into a (M+1) x N matrix
        EqnMEqualsTwo = np.array([EqnKEqualsZero, EqnKEqualsOne, EqnKEqualsTwo])
        EqnM = EqnMEqualsTwo.flatten()
    return EqnM

In [6]:
def GeneralBurgHierachy(
    u, t, delta_x, M, N,
    u_from_lower=None, t_grid=None
):
    """
    If there is no results from lower M-system (Normal):
        Evolve all coefficients u_0, ..., u_M.

    If there are results from lower M-system (Recursive):
        Keep u_0, ..., u_{M-1} fixed to the solution of 
        the (M-1) system, while evolving only u_M.

    Parameters
    ----------
    u : array
        Normal: flattened array of length (M+1)*N.
        Recursive: current u_M array of length N.

    t : vector
        time vector.

    delta_x : float
        Step size in the x-axis. 
        Can also be understood as the spacing between each grid point

    M : int
        Polynomial chaos truncation order.
        For now, this model is limited to M = 0,1,2

    N : int (non-negative)
        Number of grid points (x-axis) in which we evaluate the solution.

    u_from_lower : array or None
        If M >= 1, the odeint result for the (M-1) system,
        with shape (len(t_grid), M*N).

    t_grid : array or None
        Time grid corresponding to u_from_lower.
    """

    # The equation helper corresponding to each coefficient.
    rhs_functions = {
        0: BurgersKEqualsZeroEqn,
        1: BurgersKEqualsOneEqn,
        2: BurgersKEqualsTwoEqn
    }

    # Solve normally
    if u_from_lower is None:
        U = np.asarray(u).reshape(M + 1, N)

        rhs = [
            rhs_functions[k](U, delta_x)
            for k in range(M + 1)
        ]

        return np.asarray(rhs).flatten()

    # Solve recursively
    if M == 0:
        raise ValueError("M=0 has no lower-order system.")

    if t_grid is None:
        raise ValueError("Provide time corresponding with the lower system.")

    #lower-order solution contains M coefficients
    u_lower = np.asarray(u_from_lower).reshape(len(t_grid), M, N)
    # Interpolate the lower-order solution to the current time t.
    U = np.empty((M + 1, N))

    for k in range(M):
        for j in range(N):
            U[k, j] = np.interp(
                t, t_grid, u_lower[:, k, j]
            )

    # Fill the last row of U, corresponding to u_{M+1}
    U[M, :] = np.asarray(u).reshape(N)

    # Evaluate the M-th equation using all coefficients
    # in the M-system, but return only du_M/dt.
    return rhs_functions[M](U, delta_x)